## 1. Imports

In [1]:
import pandas as pd
import joblib

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score,
)

from sklearn.model_selection import (
    TimeSeriesSplit, 
    GridSearchCV,
)
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from xgboost import XGBRegressor

## 2. Load data

In [2]:
DF_PATH = Path("../data/processed/hour_feature_engineered.parquet")
df = pd.read_parquet(DF_PATH)

# look at the DataFrame
df.head()

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,...,year,day,hour_sin,hour_cos,month_sin,month_cos,cnt_lag_1,cnt_lag_24,cnt_lag_168,cnt
0,2011-01-08,1,0,1,7,0,6,0,2,0.16,...,2011,8,0.965926,-0.258819,0.5,0.866025,2.0,84.0,16.0,9
1,2011-01-08,1,0,1,8,0,6,0,3,0.16,...,2011,8,0.866025,-0.500000,0.5,0.866025,9.0,210.0,40.0,15
2,2011-01-08,1,0,1,9,0,6,0,3,0.16,...,2011,8,0.707107,-0.707107,0.5,0.866025,15.0,134.0,32.0,20
3,2011-01-08,1,0,1,10,0,6,0,2,0.18,...,2011,8,0.500000,-0.866025,0.5,0.866025,20.0,63.0,13.0,61
4,2011-01-08,1,0,1,11,0,6,0,2,0.20,...,2011,8,0.258819,-0.965926,0.5,0.866025,61.0,67.0,1.0,62


## 3. Train test split

In [3]:
train = df[(df["dteday"] >= pd.Timestamp(2011, 1, 8)) & (df["dteday"] <= pd.Timestamp(2012, 9, 30))]
valid = df[df["dteday"] > pd.Timestamp(2012, 9, 30)]

X_train = train.drop(columns=["dteday", "cnt"])
y_train = train["cnt"]
X_test = valid.drop(columns=["dteday", "cnt"])
y_test = valid["cnt"]

## 4. Identify columns

In [4]:
numerical_columns = ["temp", "atemp", "hum", "windspeed", "day", "hour_sin", 
                     "hour_cos", "month_sin", "month_cos", "cnt_lag_1", "cnt_lag_24", "cnt_lag_168"]
categorical_columns = ["season", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]

print(f"Numerical columns count: {len(numerical_columns)}")
print(f"Categorical columns count: {len(categorical_columns)}")

Numerical columns count: 12
Categorical columns count: 7


## 5. Build preprocessing pipelines

In [5]:
num_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
    ]
)

cat_pipeline = Pipeline(
    steps=[
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

## 6. ColumnTransformer

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", num_pipeline, numerical_columns),
        ("categorical", cat_pipeline, categorical_columns),
    ]
)

## 7. Create XG Boost model

In [7]:
xg_boost_model = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    objective="reg:squarederror"
)

## 8. Create XG Boost pipeline

In [8]:
xg_boost_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", xg_boost_model),
    ]
)

## 9. Define parameter grid

In [9]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__learning_rate": [0.01, 0.05, 0.1],
    "model__max_depth": [3, 5, 7],
    "model__subsample": [0.5, 1]
}

## 10. Fit the model

In [10]:
tscv = TimeSeriesSplit(n_splits=5)
grid = GridSearchCV(xg_boost_pipeline, param_grid, cv=tscv, scoring="r2", verbose=2)
model_grid = grid.fit(X_train, y_train)

print(f"Best hyperparameters: {model_grid.best_params_}")
print(f"Best CV score R²: {model_grid.best_score_}")

Fitting 5 folds for each of 36 candidates, totalling 180 fits
[CV] END model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__subsample=0.5; total time=   0.1s
[CV] END model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__subsample=0.5; total time=   0.0s
[CV] END model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__subsample=0.5; total time=   0.0s
[CV] END model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__subsample=0.5; total time=   0.0s
[CV] END model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__subsample=0.5; total time=   0.0s
[CV] END model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__subsample=1; total time=   0.0s
[CV] END model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__subsample=1; total time=   0.0s
[CV] END model__learning_rate=0.01, model__max_depth=3, model__n_estimators=100, model__

## 11. Calculate metrics

In [11]:
y_pred = model_grid.predict(X_test)

metrics_df = pd.DataFrame([{
    "Model Name": "XG Boost Tuned",
    "CV Mean R² score": model_grid.best_score_,
    "CV Std": model_grid.cv_results_["std_test_score"][model_grid.best_index_],
    "Test R² score": r2_score(y_test, y_pred),
    "MAE": mean_absolute_error(y_test, y_pred),
    "RMSE": root_mean_squared_error(y_test, y_pred),
}]).round(2)

# look at the metrics
metrics_df

,Model Name,CV Mean R² score,CV Std,Test R² score,MAE,RMSE
0,XG Boost Tuned,0.91,0.02,0.95,29.08,45.92


# 12. Save metrics and cv results

In [12]:
# metrics
METRICS_PATH = Path("../data/processed/best_model_metrics_df.csv")
metrics_df.to_csv(METRICS_PATH, index=False)
# cv results
CV_RESULTS_PATH = Path("../data/processed/best_model_cv_results.csv")
cv_results_df = pd.DataFrame(model_grid.cv_results_)
cv_results_df.to_csv(CV_RESULTS_PATH, index=False)

## 13. Save the best model

In [13]:
BEST_MODEL_PATH = Path("../models/xg_boost_tuned.pkl")
joblib.dump(model_grid.best_estimator_, BEST_MODEL_PATH)

['..\\models\\xg_boost_tuned.pkl']